In [1]:
"""
Sample Parse Trees Generator
CS 133 - Construction of a Context-Free Grammar for Kankanaey Simple Sentences

This script assembles the complete context-free grammar presented in Chapter III
(Context-Free Grammar Rules) -- the phrase-structure rules plus a lexicon built
directly from the POS-tagged alignment spreadsheet -- and uses it to parse a
handful of sample Kankanaey sentences, producing both a text parse tree and a
PNG tree diagram for each one, for use in the Sample Parse Trees section.

Requirements:
    pip install nltk pandas openpyxl matplotlib

Usage:
    python generate_parse_trees.py

Input file expected (same folder, or edit ALIGNMENT_FILE below):
    kankanaey_pos_alignment_template.xlsx
    columns: sentence_id, kankanaey_sentence, english_sentence,
             kankanaey_token, assigned_pos_tag
"""

import nltk
from nltk import CFG, Tree
from nltk.parse import ChartParser
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch


# ------------------------------------------------------------------
# 1. Phrase-structure rules (Section: Context-Free Grammar Rules, P)
# ------------------------------------------------------------------
PHRASE_STRUCTURE_RULES = """
S      -> SENT PUNCT
SENT   -> PRED
SENT   -> NEGP PRED
SENT   -> PRON PRED
SENT   -> ADVP PRED
SENT   -> SENT SCONJ SENT
SENT   -> SENT CCONJ SENT
PRED   -> VP
PRED   -> ADJP
VP     -> VERB
VP     -> VERB ARGS
ARGS   -> NP
ARGS   -> PP
ARGS   -> PRON
ARGS   -> NP NP
ARGS   -> NP PP
ARGS   -> NP PRON
ARGS   -> PRON NP
ARGS   -> PRON PP
ARGS   -> NUM LINK NOUN
ARGS   -> ARGS ADVP
ADJP   -> ADJ NP
ADJP   -> ADJ
NP     -> DET NOUN
NP     -> NOUN
NP     -> PROPN
NP     -> PRON
NP     -> NP NP
NP     -> NP CCONJ NP
PP     -> ADP NP
PP     -> ADP NOUN
NEGP   -> PART
NEGP   -> AUX
ADVP   -> ADV
LINK   -> PART
"""

# Only these tags are treated as valid pre-terminals. Rows in the alignment
# file whose assigned_pos_tag is anything else (e.g. an ambiguous "VERB / ADJ"
# double-tag) are dropped when building the lexicon -- see the POS Tagging
# Results section for how these ambiguous tokens were counted separately.
VALID_TAGS = {
    "VERB", "NOUN", "ADJ", "DET", "PRON", "ADP", "ADV", "PART",
    "NUM", "CCONJ", "SCONJ", "PROPN", "AUX", "INTJ", "PUNCT",
}


# ------------------------------------------------------------------
# 2. Build lexical production rules from the POS-tagged spreadsheet
# ------------------------------------------------------------------
def build_lexical_rules(alignment_path):
    """
    Reads the alignment spreadsheet and returns a string of NLTK-format
    lexical production rules, one line per POS tag, e.g.:
        VERB -> "Laylaydek" | "Inmey" | "Mankanta" | ...
    """
    df = pd.read_excel(alignment_path)
    df = df.dropna(subset=["kankanaey_token", "assigned_pos_tag"])
    df["assigned_pos_tag"] = df["assigned_pos_tag"].astype(str).str.strip()
    df = df[df["assigned_pos_tag"].isin(VALID_TAGS)]

    rules = []
    for tag, group in df.groupby("assigned_pos_tag"):
        words = sorted(set(group["kankanaey_token"]))
        # escape any double quotes inside a token, then quote it for NLTK
        quoted = " | ".join('"{}"'.format(w.replace('"', '\\"')) for w in words)
        rules.append("{} -> {}".format(tag, quoted))
    return "\n".join(rules)


# ------------------------------------------------------------------
# 3. Assemble the full grammar and build the parser
# ------------------------------------------------------------------
def build_parser(alignment_path):
    lexical_rules = build_lexical_rules(alignment_path)
    full_grammar_text = PHRASE_STRUCTURE_RULES + "\n" + lexical_rules
    grammar = CFG.fromstring(full_grammar_text)
    return ChartParser(grammar)


# ------------------------------------------------------------------
# 4. Tokenize a raw Kankanaey sentence the same way it appears in the corpus
# ------------------------------------------------------------------
def tokenize(sentence_text):
    text = sentence_text
    for punct in [".", ",", "?", "!", ";"]:
        text = text.replace(punct, " " + punct)
    return text.split()


# ------------------------------------------------------------------
# 5. Parse a sentence and display / save its parse tree
# ------------------------------------------------------------------
def parse_sentence(parser, sentence_text, save_path=None):
    tokens = tokenize(sentence_text)
    print("\nParsing: {}".format(sentence_text))
    print("Tokens:  {}".format(tokens))

    trees = list(parser.parse(tokens))
    if not trees:
        print("  -> No parse found for this sentence with the current grammar.")
        return None

    tree = trees[0]
    print(tree.pformat(margin=100))

    if save_path:
        draw_tree_image(tree, save_path)
        print("  -> Saved tree diagram to {}".format(save_path))

    if len(trees) > 1:
        print("  -> Note: {} alternative parses were found; only the first is shown/saved.".format(len(trees)))

    return tree


# ------------------------------------------------------------------
# 6. Headless tree-diagram renderer (no Tkinter / display required,
#    so this also runs cleanly in Google Colab)
# ------------------------------------------------------------------
def draw_tree_image(tree, save_path, node_color="#dbe9f7", leaf_color="#ffffff",
                     edge_color="#2c3e50"):
    positions = {}

    def assign_positions(node, depth=0, x_counter=[0]):
        if isinstance(node, Tree):
            child_xs = [assign_positions(child, depth + 1, x_counter) for child in node]
            x = sum(child_xs) / len(child_xs)
        else:
            x = x_counter[0]
            x_counter[0] += 1
        positions[id(node)] = (x, -depth)
        return x

    assign_positions(tree)

    fig, ax = plt.subplots(figsize=(max(6, len(tree.leaves()) * 1.3), 6))

    def draw_node(node):
        x, y = positions[id(node)]
        is_internal = isinstance(node, Tree)
        label = node.label() if is_internal else node
        ax.add_patch(FancyBboxPatch(
            (x - 0.45, y - 0.2), 0.9, 0.4,
            boxstyle="round,pad=0.05",
            linewidth=1, edgecolor=edge_color,
            facecolor=node_color if is_internal else leaf_color,
        ))
        ax.text(x, y, label, ha="center", va="center", fontsize=9)
        if is_internal:
            for child in node:
                cx, cy = positions[id(child)]
                ax.plot([x, cx], [y - 0.2, cy + 0.2], color=edge_color, linewidth=1)
                draw_node(child)

    draw_node(tree)
    ax.axis("off")
    xs = [x for x, y in positions.values()]
    ys = [y for x, y in positions.values()]
    ax.set_xlim(min(xs) - 1, max(xs) + 1)
    ax.set_ylim(min(ys) - 1, 1)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close(fig)


# ------------------------------------------------------------------
# 7. Run on the sample sentences used in the Sample Parse Trees section
# ------------------------------------------------------------------
if __name__ == "__main__":
    ALIGNMENT_FILE = "kankanaey_pos_alignment_template.xlsx" # edit path if needed

    parser = build_parser(ALIGNMENT_FILE)

    sample_sentences = [
        "Inmey din lalaki.",
        "Mankanta din babai.",
        "Mapo daida sin liyang.",
    ]

    for i, sentence in enumerate(sample_sentences, start=1):
        parse_sentence(parser, sentence, save_path="parse_tree_{}.png".format(i))

FileNotFoundError: [Errno 2] No such file or directory: 'main/data/kankanaey_pos_alignment_template.xlsx'